# BM25 Search Baseline

This notebook establishes a lexical BM25 baseline for the product search system.

BM25 retrieves products based on lexical overlap between the user's search query
and product text. This provides a strong traditional information-retrieval baseline
before moving to dense retrieval, hybrid search, and learning-to-rank.

The product catalog contains 124,428 products. Model development and evaluation
use query-level train/validation/test splits to prevent query leakage.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

In [5]:
PROJECT_ROOT = Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PRODUCT_CATALOG_PATH = PROCESSED_DIR / "product_catalog.parquet"
TRAIN_PATH = PROCESSED_DIR / "train.parquet"
VALIDATION_PATH = PROCESSED_DIR / "validation.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

In [7]:
product_catalog = pd.read_parquet(PRODUCT_CATALOG_PATH)
validation = pd.read_parquet(VALIDATION_PATH)

In [14]:
def tokenize(text):
    return text.lower().split()

In [17]:
product_tokens = product_catalog["product_text"].map(tokenize)

In [18]:
product_tokens.head(1).iloc[0][:5]

['title:', 'simpson', 'strong-tie', '12-gauge', 'angle.']

In [19]:
bm25 = BM25Okapi(product_tokens)

In [20]:
query = "cordless drill"

query_tokens = tokenize(query)

scores = bm25.get_scores(query_tokens)

top_k = 10

top_indices = np.argsort(scores)[::-1][:top_k]

results = product_catalog.iloc[top_indices][
    ["product_uid", "product_title"]
].copy()

results["bm25_score"] = scores[top_indices]

results

,product_uid,product_title,bm25_score
39794,139795,Husky Cordless Drill Holder,16.325985
96602,196603,Hilti Belt Hook for Cordless Drill and Impact,16.254412
63290,163291,Hilti 18-Volt Lithium-Ion Cordless Rotary Hamm...,15.921617
46066,146067,Hilti 18-Volt Lithium-Ion Cordless Rotary Hamm...,15.876192
33590,133591,Hilti 18-Volt Lithium-Ion Cordless Rotary Hamm...,15.693337
13389,113390,Milwaukee M12 12-Volt Lithium-Ion Cordless Dri...,15.571870
96673,196674,BLACK+DECKER 18-Volt NiCad Project Kit with Co...,15.350605
48161,148162,Milwaukee M18 18-Volt Lithium-Ion 1/2 in. Cord...,15.341076
54344,154345,Milwaukee M12 12-Volt Lithium-Ion Cordless Dri...,15.271396
6960,106961,BLACK+DECKER 12-Volt Lithium-Ion Cordless Dril...,15.215763


In [21]:
from sklearn.metrics import ndcg_score

In [22]:
catalog_lookup = product_catalog.set_index("product_uid")

In [24]:
missing_products = ~validation["product_uid"].isin(product_catalog["product_uid"])

print("Missing products:", missing_products.sum())

Missing products: 0


In [25]:
product_index = pd.Series(
    np.arange(len(product_catalog)),
    index=product_catalog["product_uid"]
)

In [27]:
def get_bm25_scores(query, product_uids):
    query_tokens = tokenize(query)

    indices = product_index.loc[product_uids].to_numpy()

    scores = bm25.get_scores(query_tokens)

    return scores[indices]

In [28]:
query = validation["search_term"].iloc[0]

query_data = validation[
    validation["search_term"] == query
].copy()

query_data

,id,product_uid,product_title,search_term,relevance
0,9,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 ...,deck over,3.00
18,571,100092,Rust-Oleum Restore 1-gal. 4X Deck Coat,deck over,1.67
301,9568,101629,BEHR Premium DeckOver 5-gal. #PFC-68 Silver Gr...,deck over,3.00
330,10428,101795,BEHR Premium DeckOver 5-gal. #SC-145 Desert Sa...,deck over,2.67
544,17058,103015,Rust-Oleum Restore 4 gal. 10X Advanced Timberl...,deck over,1.67
550,17297,103057,Rust-Oleum Restore 5-gal. 4X Deck Coat,deck over,2.33
604,18775,103356,BEHR Premium DeckOver 5-gal. #SC-330 Redwood W...,deck over,2.67
609,18899,103384,Rust-Oleum Restore 4 gal. 10X Advanced Cape Co...,deck over,2.33


In [29]:
scores = get_bm25_scores(
    query,
    query_data["product_uid"].tolist()
)

query_data["bm25_score"] = scores

query_data.sort_values(
    "bm25_score",
    ascending=False
)

,id,product_uid,product_title,search_term,relevance,bm25_score
544,17058,103015,Rust-Oleum Restore 4 gal. 10X Advanced Timberl...,deck over,1.67,6.924154
609,18899,103384,Rust-Oleum Restore 4 gal. 10X Advanced Cape Co...,deck over,2.33,6.914115
18,571,100092,Rust-Oleum Restore 1-gal. 4X Deck Coat,deck over,1.67,6.677194
550,17297,103057,Rust-Oleum Restore 5-gal. 4X Deck Coat,deck over,2.33,6.677194
0,9,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 ...,deck over,3.00,2.790315
301,9568,101629,BEHR Premium DeckOver 5-gal. #PFC-68 Silver Gr...,deck over,3.00,0.000000
330,10428,101795,BEHR Premium DeckOver 5-gal. #SC-145 Desert Sa...,deck over,2.67,0.000000
604,18775,103356,BEHR Premium DeckOver 5-gal. #SC-330 Redwood W...,deck over,2.67,0.000000


In [32]:
def calculate_ndcg(query_data, k):
    y_true = query_data["relevance"].to_numpy()
    y_score = query_data["bm25_score"].to_numpy()

    return ndcg_score(
        [y_true],
        [y_score],
        k=k
    )

In [33]:
print("NDCG@10:", calculate_ndcg(query_data, 10))
print("NDCG@20:", calculate_ndcg(query_data, 20))

NDCG@10: 0.8786732048269063
NDCG@20: 0.8786732048269063


In [34]:
results = []

for query, query_data in validation.groupby("search_term"):
    query_data = query_data.copy()

    query_data["bm25_score"] = get_bm25_scores(
        query,
        query_data["product_uid"].tolist()
    )

    results.append({
        "query": query,
        "num_products": len(query_data),
        "ndcg@10": calculate_ndcg(query_data, 10),
        "ndcg@20": calculate_ndcg(query_data, 20),
    })

bm25_results = pd.DataFrame(results)

In [35]:
bm25_results.head()

,query,num_products,ndcg@10,ndcg@20
0,.75 in pocket hole screws,5,0.930503,0.930503
1,"0.5mm bit, counter sink ground",4,0.892776,0.892776
2,1 1/4 pvcsch 80,9,0.936231,0.936231
3,1 in to 3/4 inch pvc elbow,4,0.978443,0.978443
4,1' pvc pipe,8,0.939936,0.939936


In [36]:
print(
    "Mean NDCG@10:",
    bm25_results["ndcg@10"].mean()
)

print(
    "Mean NDCG@20:",
    bm25_results["ndcg@20"].mean()
)

Mean NDCG@10: 0.9532332010921856
Mean NDCG@20: 0.9567022882702825


In [37]:
bm25_results[
    ["ndcg@10", "ndcg@20"]
].describe()

,ndcg@10,ndcg@20
count,1040.000000,1040.000000
mean,0.953233,0.956702
std,0.041255,0.037971
min,0.716711,0.798947
25%,0.933375,0.937771
50%,0.961386,0.964441
75%,0.985588,0.986895
max,1.000000,1.000000


In [39]:
relevance_counts = (
    validation
    .assign(is_relevant=validation["relevance"] >= 2.0) # set a threshold for relevance
    .groupby("search_term")["is_relevant"]
    .sum()
)

print(relevance_counts.describe())

count    1040.000000
mean        6.001923
std         2.965311
min         0.000000
25%         4.000000
50%         6.000000
75%         8.000000
max        15.000000
Name: is_relevant, dtype: float64


In [40]:
print(
    "Queries with at least 1 relevant product:",
    (relevance_counts > 0).sum()
)

print(
    "Queries with 2+ relevant products:",
    (relevance_counts >= 2).sum()
)

print(
    "Queries with 3+ relevant products:",
    (relevance_counts >= 3).sum()
)

Queries with at least 1 relevant product: 1031
Queries with 2+ relevant products: 995
Queries with 3+ relevant products: 891


In [41]:
def get_top_k_products(query, k=100):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    return product_catalog.iloc[top_indices][
        ["product_uid", "product_title"]
    ].copy()

In [42]:
def recall_at_k(retrieved_uids, relevant_uids, k):
    retrieved = set(retrieved_uids[:k])
    relevant = set(relevant_uids)

    if not relevant:
        return np.nan

    return len(retrieved & relevant) / len(relevant)

In [43]:
relevance_counts = (
    validation
    .assign(is_relevant=validation["relevance"] >= 2.0)
    .groupby("search_term")["is_relevant"]
    .sum()
)

print(relevance_counts.describe())

print(
    "Queries with at least 1 relevant product:",
    (relevance_counts > 0).sum()
)

print(
    "Queries with 2+ relevant products:",
    (relevance_counts >= 2).sum()
)

print(
    "Queries with 3+ relevant products:",
    (relevance_counts >= 3).sum()
)

count    1040.000000
mean        6.001923
std         2.965311
min         0.000000
25%         4.000000
50%         6.000000
75%         8.000000
max        15.000000
Name: is_relevant, dtype: float64
Queries with at least 1 relevant product: 1031
Queries with 2+ relevant products: 995
Queries with 3+ relevant products: 891


In [44]:
def retrieve_top_k(query, k=100):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    return product_catalog.iloc[top_indices]["product_uid"].to_numpy()

In [47]:
from tqdm.auto import tqdm
recall_results = []

for query, query_data in tqdm(
    validation.groupby("search_term"),
    total=validation["search_term"].nunique()
):

    relevant_uids = query_data.loc[
        query_data["relevance"] >= 2.0,
        "product_uid"
    ].to_numpy()

    retrieved_uids = retrieve_top_k(query, k=100)

    recall_results.append({
        "query": query,
        "num_relevant": len(relevant_uids),
        "recall@10": recall_at_k(
            retrieved_uids, relevant_uids, 10
        ),
        "recall@20": recall_at_k(
            retrieved_uids, relevant_uids, 20
        ),
        "recall@50": recall_at_k(
            retrieved_uids, relevant_uids, 50
        ),
        "recall@100": recall_at_k(
            retrieved_uids, relevant_uids, 100
        ),
    })

c:\Users\Kevin\Documents\GitHub\home-depot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1040/1040 [04:24<00:00,  3.93it/s]


In [48]:
recall_results = pd.DataFrame(recall_results)

In [49]:
recall_results[
    ["recall@10", "recall@20", "recall@50", "recall@100"]
].describe()

,recall@10,recall@20,recall@50,recall@100
count,1031.000000,1031.000000,1031.000000,1031.000000
mean,0.197281,0.282975,0.401176,0.491679
std,0.276281,0.332617,0.376272,0.388612
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.100000
50%,0.000000,0.142857,0.333333,0.500000
75%,0.333333,0.500000,0.750000,0.888889
max,1.000000,1.000000,1.000000,1.000000
